# Sendi et al. (2025) Replication

This notebook replicates the methodology of Sendi et al. (2025) for comparison with the proposed returns-based approach in the main dissertation.

**Replication methodology (per their paper):**
- Data range: July 2019 – July 2023 (4 years)
- Five tickers: AAPL, CSCO, META, MSFT, TSLA
- Architectures: GRU, LSTM, Transformer (each 64/64)
- Normalisation: MinMaxScaler on raw prices
- Train/val/test split: 80/10/10

**Cells:**
- **Cell 1**: Core replication — GRU/LSTM/Transformer across 5 tickers
- **Cell 2**: Augmentation experiments (supports Experiment 8 in dissertation)
- **Cell 3**: Classification reformulation experiments (supports Experiment 8 / Table XXII)

**Reported numbers (AAPL, used in Table XXIV of dissertation):**
- GRU RMSE ≈ $2.72 ± $0.30
- LSTM RMSE ≈ $3.54 ± $1.09

See the dissertation footnote to Table XXIV for discussion of how Sendi et al.'s original reported RMSE (1.56/1.59) relates to these dollar-scale values.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, GRU, Dense, Dropout, MultiHeadAttention,
    LayerNormalization, GlobalAveragePooling1D, Input
)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import yfinance as yf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# SENDI REPLICATION — corrected to match the paper
#
# Key corrections vs. original script:
#   1. Date range: July 2019 – July 2023 (not 2013–2025)
#   2. Split: 80/10/10 (not 70/15/15)
#   3. Architecture: 64/64 for GRU & LSTM (not 64/32)
#   4. Transformer: 3 encoder blocks (not 2)
#   5. All 5 tickers: AAPL, CSCO, META, MSFT, TSLA
#
# Two approaches retained:
#   A) Sendi's methodology (raw prices, MinMaxScaler)
#   B) Returns-based methodology with same architectures
# ============================================================

TICKERS = ['AAPL', 'CSCO', 'META', 'MSFT', 'TSLA']
START_DATE = '2019-07-01'
END_DATE = '2023-07-01'
WINDOW = 30  # paper doesn't specify; 30 is a common default

# ============================================================
# APPROACH A: Sendi's methodology (raw prices, MinMaxScaler)
# ============================================================

def download_data(ticker):
    df = yf.download(ticker, start=START_DATE, end=END_DATE, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df

def create_seq_univariate(data, window):
    X, y = [], []
    for i in range(window, len(data)):
        X.append(data[i-window:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

def make_gru_sendi():
    """GRU: two layers of 64 units each (matching paper)."""
    return Sequential([
        GRU(64, return_sequences=True, input_shape=(WINDOW, 1)),
        Dropout(0.2),
        GRU(64),
        Dropout(0.2),
        Dense(1)
    ])

def make_lstm_sendi():
    """LSTM: two layers of 64 units each (matching paper)."""
    return Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW, 1)),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(1)
    ])

def make_transformer_sendi():
    """Transformer: 3 encoder blocks with multi-head attention (matching paper)."""
    inputs = Input(shape=(WINDOW, 1))
    x = Dense(64)(inputs)  # project to d_model=64

    # Encoder block 1
    attn1 = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = LayerNormalization()(x + attn1)
    ff1 = Dense(128, activation='relu')(x)
    ff1 = Dense(64)(ff1)
    x = LayerNormalization()(x + ff1)

    # Encoder block 2
    attn2 = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = LayerNormalization()(x + attn2)
    ff2 = Dense(128, activation='relu')(x)
    ff2 = Dense(64)(ff2)
    x = LayerNormalization()(x + ff2)

    # Encoder block 3
    attn3 = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
    x = LayerNormalization()(x + attn3)
    ff3 = Dense(128, activation='relu')(x)
    ff3 = Dense(64)(ff3)
    x = LayerNormalization()(x + ff3)

    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.2)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    outputs = Dense(1)(x)

    return keras.Model(inputs, outputs)

def run_model_sendi(model_fn, name, X_tr, y_tr, X_v, y_v, X_te, y_te, scaler, runs=10):
    all_rmse, all_mae, all_r2 = [], [], []
    all_rmse_scaled = []

    for run in range(runs):
        seed = 100 + run
        tf.random.set_seed(seed)
        np.random.seed(seed)

        model = model_fn()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mse')
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
                  epochs=200, batch_size=32, callbacks=[es], verbose=0)

        preds_scaled = model.predict(X_te, verbose=0).ravel()
        actual_scaled = y_te

        # Scaled RMSE (what Sendi likely reported)
        rmse_s = np.sqrt(np.mean((preds_scaled - actual_scaled)**2))
        all_rmse_scaled.append(rmse_s)

        # Inverse transform to dollar prices
        preds_dollar = scaler.inverse_transform(preds_scaled.reshape(-1, 1)).ravel()
        actual_dollar = scaler.inverse_transform(actual_scaled.reshape(-1, 1)).ravel()

        rmse = np.sqrt(np.mean((preds_dollar - actual_dollar)**2))
        mae = np.mean(np.abs(preds_dollar - actual_dollar))

        # R² (Sendi reported this)
        ss_res = np.sum((actual_dollar - preds_dollar)**2)
        ss_tot = np.sum((actual_dollar - np.mean(actual_dollar))**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

        all_rmse.append(rmse)
        all_mae.append(mae)
        all_r2.append(r2)

        epochs_used = len(model.history.history['loss'])
        print(f"  Run {run+1:2d}: RMSE=${rmse:.2f}  MAE=${mae:.2f}  R²={r2:.4f}  Epochs={epochs_used}")

    params = model.count_params()
    print(f"\n  SUMMARY — {name}:")
    print(f"  RMSE (dollar):  ${np.mean(all_rmse):.2f} ± {np.std(all_rmse):.2f}")
    print(f"  MAE  (dollar):  ${np.mean(all_mae):.2f} ± {np.std(all_mae):.2f}")
    print(f"  R²:             {np.mean(all_r2):.4f} ± {np.std(all_r2):.4f}")
    print(f"  RMSE (scaled):  {np.mean(all_rmse_scaled):.5f} ± {np.std(all_rmse_scaled):.5f}")
    print(f"  Parameters:     {params}")
    return {
        'rmse': np.mean(all_rmse), 'rmse_std': np.std(all_rmse),
        'mae': np.mean(all_mae), 'mae_std': np.std(all_mae),
        'r2': np.mean(all_r2), 'r2_std': np.std(all_r2),
        'rmse_scaled': np.mean(all_rmse_scaled), 'rmse_scaled_std': np.std(all_rmse_scaled),
        'params': params
    }

# ============================================================
# APPROACH B: Returns-based methodology
# ============================================================

FEATURES = [
    'daily_return', 'ret_lag_2', 'ret_lag_5', 'ret_lag_10',
    'close_ma10_ratio', 'close_ma30_ratio',
    'volatility_10', 'volatility_30',
    'vol_change', 'oc_gap'
]

def engineer_features(df):
    df2 = df.copy()
    df2['daily_return'] = df2['Close'].pct_change(1)
    df2['ret_lag_2'] = df2['Close'].pct_change(2)
    df2['ret_lag_5'] = df2['Close'].pct_change(5)
    df2['ret_lag_10'] = df2['Close'].pct_change(10)
    df2['close_ma10_ratio'] = df2['Close'] / df2['Close'].rolling(10).mean() - 1
    df2['close_ma30_ratio'] = df2['Close'] / df2['Close'].rolling(30).mean() - 1
    df2['volatility_10'] = df2['Close'].pct_change(1).rolling(10).std()
    df2['volatility_30'] = df2['Close'].pct_change(1).rolling(30).std()
    df2['vol_change'] = df2['Volume'].pct_change(1)
    df2['oc_gap'] = (df2['Close'] - df2['Open']) / df2['Open']
    df2['target_return'] = df2['Close'].pct_change(1).shift(-1)
    df2.dropna(inplace=True)
    return df2

def create_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

def make_gru_returns():
    return Sequential([
        GRU(64, return_sequences=True, input_shape=(WINDOW, len(FEATURES)),
            kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005)),
        Dropout(0.3),
        GRU(64, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005)),
        Dropout(0.3),
        Dense(1)
    ])

def make_lstm_returns():
    return Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW, len(FEATURES)),
             kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005)),
        Dropout(0.3),
        LSTM(64, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005)),
        Dropout(0.3),
        Dense(1)
    ])

def make_transformer_returns():
    inputs = Input(shape=(WINDOW, len(FEATURES)))
    x = Dense(64)(inputs)

    for _ in range(3):
        attn = MultiHeadAttention(num_heads=4, key_dim=16)(x, x)
        x = LayerNormalization()(x + attn)
        ff = Dense(128, activation='relu')(x)
        ff = Dense(64)(ff)
        x = LayerNormalization()(x + ff)

    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1)(x)
    return keras.Model(inputs, outputs)

def run_model_returns(model_fn, name, X_train, y_train, X_val, y_val,
                      X_test, y_test, scaler_y, test_prices, test_returns_raw, runs=10):
    all_rmse, all_mae, all_da = [], [], []

    for run in range(runs):
        seed = 100 + run
        tf.random.set_seed(seed)
        np.random.seed(seed)

        model = model_fn()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='mse')
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=200, batch_size=32, callbacks=[es], verbose=0)

        preds_s = model.predict(X_test, verbose=0).ravel()
        preds_ret = scaler_y.inverse_transform(preds_s.reshape(-1, 1)).ravel()
        preds_price = test_prices * (1 + preds_ret)
        actual_price = test_prices * (1 + test_returns_raw)

        rmse = np.sqrt(np.mean((preds_price - actual_price)**2))
        mae = np.mean(np.abs(preds_price - actual_price))
        da = np.mean(np.sign(preds_ret) == np.sign(test_returns_raw)) * 100

        all_rmse.append(rmse)
        all_mae.append(mae)
        all_da.append(da)

        epochs_used = len(model.history.history['loss'])
        print(f"  Run {run+1:2d}: RMSE=${rmse:.2f}  MAE=${mae:.2f}  DA={da:.1f}%  Epochs={epochs_used}")

    params = model.count_params()
    n_test = len(X_test)
    n_correct = int(round(np.mean(all_da) / 100 * n_test))
    from scipy.stats import binomtest
    p_val = binomtest(n_correct, n_test, 0.5, alternative='greater').pvalue

    print(f"\n  SUMMARY — {name}:")
    print(f"  RMSE: ${np.mean(all_rmse):.2f} ± {np.std(all_rmse):.2f}")
    print(f"  MAE:  ${np.mean(all_mae):.2f} ± {np.std(all_mae):.2f}")
    print(f"  DA:   {np.mean(all_da):.1f}% ± {np.std(all_da):.1f}%")
    print(f"  DA p-value: {p_val:.2e}")
    print(f"  Parameters: {params}")
    return {
        'rmse': np.mean(all_rmse), 'rmse_std': np.std(all_rmse),
        'mae': np.mean(all_mae), 'mae_std': np.std(all_mae),
        'da': np.mean(all_da), 'da_std': np.std(all_da),
        'p_val': p_val, 'params': params
    }


# ============================================================
# MAIN LOOP — run all tickers × all models
# ============================================================

results_a = {}  # Approach A results
results_b = {}  # Approach B results

for ticker in TICKERS:
    print("\n" + "#"*80)
    print(f"#  TICKER: {ticker}")
    print("#"*80)

    df = download_data(ticker)
    if len(df) < WINDOW + 50:
        print(f"  Skipping {ticker} — insufficient data")
        continue

    # ----------------------------------------------------------
    # APPROACH A: Sendi replication
    # ----------------------------------------------------------
    print("\n" + "="*70)
    print(f"  APPROACH A: Sendi methodology — {ticker}")
    print("="*70)

    close_prices = df['Close'].values.reshape(-1, 1)
    n = len(close_prices)
    train_end = int(n * 0.80)
    val_end = int(n * 0.90)

    scaler_mm = MinMaxScaler().fit(close_prices[:train_end])
    scaled = scaler_mm.transform(close_prices)

    X_all, y_all = create_seq_univariate(scaled, WINDOW)
    X_all = X_all.reshape(-1, WINDOW, 1)

    train_end_seq = train_end - WINDOW
    val_end_seq = val_end - WINDOW

    X_tr_a = X_all[:train_end_seq]
    y_tr_a = y_all[:train_end_seq]
    X_va_a = X_all[train_end_seq:val_end_seq]
    y_va_a = y_all[train_end_seq:val_end_seq]
    X_te_a = X_all[val_end_seq:]
    y_te_a = y_all[val_end_seq:]

    print(f"  Samples — Train: {len(X_tr_a)}, Val: {len(X_va_a)}, Test: {len(X_te_a)}")

    results_a[ticker] = {}

    print(f"\n--- GRU (Sendi) — {ticker} ---")
    results_a[ticker]['GRU'] = run_model_sendi(
        make_gru_sendi, f"GRU Sendi ({ticker})",
        X_tr_a, y_tr_a, X_va_a, y_va_a, X_te_a, y_te_a, scaler_mm, runs=10)

    print(f"\n--- LSTM (Sendi) — {ticker} ---")
    results_a[ticker]['LSTM'] = run_model_sendi(
        make_lstm_sendi, f"LSTM Sendi ({ticker})",
        X_tr_a, y_tr_a, X_va_a, y_va_a, X_te_a, y_te_a, scaler_mm, runs=10)

    print(f"\n--- Transformer (Sendi) — {ticker} ---")
    results_a[ticker]['Transformer'] = run_model_sendi(
        make_transformer_sendi, f"Transformer Sendi ({ticker})",
        X_tr_a, y_tr_a, X_va_a, y_va_a, X_te_a, y_te_a, scaler_mm, runs=10)

    # ----------------------------------------------------------
    # APPROACH B: Returns-based
    # ----------------------------------------------------------
    print("\n" + "="*70)
    print(f"  APPROACH B: Returns methodology — {ticker}")
    print("="*70)

    df2 = engineer_features(df)
    X_all_b = df2[FEATURES].values
    y_all_b = df2['target_return'].values
    prices_b = df2['Close'].values

    n_b = len(X_all_b)
    train_end_b = int(n_b * 0.80)
    val_end_b = int(n_b * 0.90)

    scaler_X = StandardScaler().fit(X_all_b[:train_end_b])
    scaler_y = StandardScaler().fit(y_all_b[:train_end_b].reshape(-1, 1))

    X_tr_s = scaler_X.transform(X_all_b[:train_end_b])
    X_va_s = scaler_X.transform(X_all_b[train_end_b:val_end_b])
    X_te_s = scaler_X.transform(X_all_b[val_end_b:])
    y_tr_s = scaler_y.transform(y_all_b[:train_end_b].reshape(-1, 1)).ravel()
    y_va_s = scaler_y.transform(y_all_b[train_end_b:val_end_b].reshape(-1, 1)).ravel()
    y_te_s = scaler_y.transform(y_all_b[val_end_b:].reshape(-1, 1)).ravel()

    X_train_b, y_train_b = create_sequences(X_tr_s, y_tr_s, WINDOW)
    X_val_b, y_val_b = create_sequences(X_va_s, y_va_s, WINDOW)
    X_test_b, y_test_b = create_sequences(X_te_s, y_te_s, WINDOW)

    test_prices_b = prices_b[val_end_b + WINDOW:]
    test_returns_b = y_all_b[val_end_b + WINDOW:]

    # Align lengths
    min_len = min(len(X_test_b), len(test_prices_b), len(test_returns_b))
    X_test_b = X_test_b[:min_len]
    y_test_b = y_test_b[:min_len]
    test_prices_b = test_prices_b[:min_len]
    test_returns_b = test_returns_b[:min_len]

    print(f"  Samples — Train: {len(X_train_b)}, Val: {len(X_val_b)}, Test: {len(X_test_b)}")

    results_b[ticker] = {}

    print(f"\n--- GRU (returns) — {ticker} ---")
    results_b[ticker]['GRU'] = run_model_returns(
        make_gru_returns, f"GRU returns ({ticker})",
        X_train_b, y_train_b, X_val_b, y_val_b,
        X_test_b, y_test_b, scaler_y, test_prices_b, test_returns_b, runs=10)

    print(f"\n--- LSTM (returns) — {ticker} ---")
    results_b[ticker]['LSTM'] = run_model_returns(
        make_lstm_returns, f"LSTM returns ({ticker})",
        X_train_b, y_train_b, X_val_b, y_val_b,
        X_test_b, y_test_b, scaler_y, test_prices_b, test_returns_b, runs=10)

    print(f"\n--- Transformer (returns) — {ticker} ---")
    results_b[ticker]['Transformer'] = run_model_returns(
        make_transformer_returns, f"Transformer returns ({ticker})",
        X_train_b, y_train_b, X_val_b, y_val_b,
        X_test_b, y_test_b, scaler_y, test_prices_b, test_returns_b, runs=10)


# ============================================================
# FINAL SUMMARY TABLES
# ============================================================
print("\n\n" + "="*80)
print("  FINAL SUMMARY — APPROACH A (Sendi replication)")
print("="*80)
print(f"{'Ticker':<8} {'Model':<14} {'RMSE($)':<14} {'MAE($)':<14} {'R²':<14} {'RMSE(scaled)':<16} {'Params'}")
print("-"*90)
for ticker in TICKERS:
    if ticker not in results_a:
        continue
    for model_name in ['GRU', 'LSTM', 'Transformer']:
        r = results_a[ticker][model_name]
        print(f"{ticker:<8} {model_name:<14} "
              f"${r['rmse']:>7.2f}±{r['rmse_std']:.2f}  "
              f"${r['mae']:>7.2f}±{r['mae_std']:.2f}  "
              f"{r['r2']:>6.4f}±{r['r2_std']:.4f}  "
              f"{r['rmse_scaled']:>8.5f}±{r['rmse_scaled_std']:.5f}  "
              f"{r['params']}")

print("\n\n" + "="*80)
print("  FINAL SUMMARY — APPROACH B (Returns-based)")
print("="*80)
print(f"{'Ticker':<8} {'Model':<14} {'RMSE($)':<14} {'MAE($)':<14} {'DA(%)':<14} {'p-value':<12} {'Params'}")
print("-"*90)
for ticker in TICKERS:
    if ticker not in results_b:
        continue
    for model_name in ['GRU', 'LSTM', 'Transformer']:
        r = results_b[ticker][model_name]
        print(f"{ticker:<8} {model_name:<14} "
              f"${r['rmse']:>7.2f}±{r['rmse_std']:.2f}  "
              f"${r['mae']:>7.2f}±{r['mae_std']:.2f}  "
              f"{r['da']:>5.1f}±{r['da_std']:.1f}%  "
              f"{r['p_val']:.2e}     "
              f"{r['params']}")


################################################################################
#  TICKER: AAPL
################################################################################

  APPROACH A: Sendi methodology — AAPL
  Samples — Train: 776, Val: 101, Test: 101

--- GRU (Sendi) — AAPL ---
  Run  1: RMSE=$2.43  MAE=$1.90  R²=0.9615  Epochs=143
  Run  2: RMSE=$2.59  MAE=$2.09  R²=0.9560  Epochs=79


  Run  3: RMSE=$3.00  MAE=$2.48  R²=0.9411  Epochs=63
  Run  4: RMSE=$3.31  MAE=$2.81  R²=0.9282  Epochs=28
  Run  5: RMSE=$2.31  MAE=$1.80  R²=0.9653  Epochs=116
  Run  6: RMSE=$2.79  MAE=$2.23  R²=0.9491  Epochs=122
  Run  7: RMSE=$2.78  MAE=$2.24  R²=0.9494  Epochs=92
  Run  8: RMSE=$2.38  MAE=$1.86  R²=0.9629  Epochs=164
  Run  9: RMSE=$2.66  MAE=$2.22  R²=0.9539  Epochs=54
  Run 10: RMSE=$2.94  MAE=$2.50  R²=0.9434  Epochs=23

  SUMMARY — GRU Sendi (AAPL):
  RMSE (dollar):  $2.72 ± 0.30
  MAE  (dollar):  $2.21 ± 0.30
  R²:             0.9511 ± 0.0108
  RMSE (scaled):  0.02064 ± 0.00225
  Parameters:     37889

--- LSTM (Sendi) — AAPL ---
  Run  1: RMSE=$2.94  MAE=$2.37  R²=0.9436  Epochs=178
  Run  2: RMSE=$3.21  MAE=$2.66  R²=0.9327  Epochs=123
  Run  3: RMSE=$2.45  MAE=$1.90  R²=0.9609  Epochs=200
  Run  4: RMSE=$6.67  MAE=$6.05  R²=0.7092  Epochs=20
  Run  5: RMSE=$3.66  MAE=$3.03  R²=0.9126  Epochs=116
  Run  6: RMSE=$3.46  MAE=$2.92  R²=0.9218  Epochs=105
  Run  7: RMSE=$2.93

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import yfinance as yf
from scipy.stats import binomtest
import warnings
warnings.filterwarnings('ignore')

df = yf.download('AAPL', start='2013-01-01', end='2025-12-31')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df['daily_return'] = df['Close'].pct_change(1)
df['ret_lag_2'] = df['Close'].pct_change(2)
df['ret_lag_5'] = df['Close'].pct_change(5)
df['ret_lag_10'] = df['Close'].pct_change(10)
df['close_ma10_ratio'] = df['Close'] / df['Close'].rolling(10).mean() - 1
df['close_ma30_ratio'] = df['Close'] / df['Close'].rolling(30).mean() - 1
df['volatility_10'] = df['Close'].pct_change(1).rolling(10).std()
df['volatility_30'] = df['Close'].pct_change(1).rolling(30).std()
df['vol_change'] = df['Volume'].pct_change(1)
df['oc_gap'] = (df['Close'] - df['Open']) / df['Open']
df['target_return'] = df['Close'].pct_change(1).shift(-1)
df.dropna(inplace=True)

FEATURES = ['daily_return','ret_lag_2','ret_lag_5','ret_lag_10',
            'close_ma10_ratio','close_ma30_ratio','volatility_10','volatility_30',
            'vol_change','oc_gap']
WINDOW = 30

X_all = df[FEATURES].values
y_all = df['target_return'].values
prices = df['Close'].values

n = len(X_all)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

scaler_X = StandardScaler().fit(X_all[:train_end])
scaler_y = StandardScaler().fit(y_all[:train_end].reshape(-1,1))

X_tr_s = scaler_X.transform(X_all[:train_end])
X_va_s = scaler_X.transform(X_all[train_end:val_end])
X_te_s = scaler_X.transform(X_all[val_end:])
y_tr_s = scaler_y.transform(y_all[:train_end].reshape(-1,1)).ravel()
y_va_s = scaler_y.transform(y_all[train_end:val_end].reshape(-1,1)).ravel()
y_te_s = scaler_y.transform(y_all[val_end:].reshape(-1,1)).ravel()

def create_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train, y_train = create_sequences(X_tr_s, y_tr_s, WINDOW)
X_val, y_val = create_sequences(X_va_s, y_va_s, WINDOW)
X_test, y_test = create_sequences(X_te_s, y_te_s, WINDOW)

test_prices = prices[val_end + WINDOW:]
test_returns = y_all[val_end + WINDOW:]
actual_direction = np.sign(test_returns)

# Unscale training targets to check direction
y_train_raw = scaler_y.inverse_transform(y_train.reshape(-1,1)).ravel()

print(f"Dataset: {len(df)} days | Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
up_days = np.sum(y_train_raw > 0)
down_days = np.sum(y_train_raw <= 0)
print(f"Training set: {up_days} up days ({up_days/len(y_train_raw)*100:.1f}%), {down_days} down days ({down_days/len(y_train_raw)*100:.1f}%)")
print(f"Test set: {np.sum(test_returns > 0)} up ({np.sum(test_returns > 0)/len(test_returns)*100:.1f}%), {np.sum(test_returns <= 0)} down ({np.sum(test_returns <= 0)/len(test_returns)*100:.1f}%)")

# ============================================================
# HELPER
# ============================================================
def evaluate_model(model, X_te, name):
    preds_s = model.predict(X_te, verbose=0).ravel()
    preds_ret = scaler_y.inverse_transform(preds_s.reshape(-1,1)).ravel()
    pred_dir = np.sign(preds_ret)

    preds_price = test_prices * (1 + preds_ret)
    actual_price = test_prices * (1 + test_returns)
    rmse = np.sqrt(np.mean((preds_price - actual_price)**2))
    mae = np.mean(np.abs(preds_price - actual_price))
    da = np.mean(pred_dir == actual_direction) * 100

    n_correct = int(round(da / 100 * len(X_te)))
    p_val = binomtest(n_correct, len(X_te), 0.5, alternative='greater').pvalue

    # Count wrong types
    wrong = pred_dir != actual_direction
    said_up_went_down = np.sum((pred_dir == 1) & (actual_direction == -1))
    said_down_went_up = np.sum((pred_dir == -1) & (actual_direction == 1))

    print(f"  {name:<45} | ${rmse:>5.2f} | {da:>5.1f}% | {p_val:.3f} | up→down:{said_up_went_down:>3} | down→up:{said_down_went_up:>3}")
    return preds_ret, pred_dir

header = f"  {'Method':<45} | {'RMSE':>6} | {'DA':>6} | {'p-val':>5} | {'Wrong Types':>25}"

# ============================================================
# AUGMENTATION TECHNIQUES
# ============================================================
print(f"\n{'='*110}")
print(f"  ADVANCED DATA AUGMENTATION TO REDUCE WRONG CLASSIFICATIONS")
print(f"{'='*110}")
print(header)
print("  " + "-"*105)

def make_model():
    inp = Input(shape=(WINDOW, len(FEATURES)))
    x = LSTM(64, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
    x = Dropout(0.3)(x)
    x = LSTM(32, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
    x = Dropout(0.3)(x)
    out = Dense(1)(x)
    return Model(inp, out)

def train_and_eval(X_tr, y_tr, name, sample_weights=None):
    all_preds = []
    for run in range(5):
        seed = 100 + run
        tf.random.set_seed(seed)
        np.random.seed(seed)
        model = make_model()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='mse')
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        if sample_weights is not None:
            model.fit(X_tr, y_tr, sample_weight=sample_weights,
                      validation_data=(X_val, y_val),
                      epochs=200, batch_size=32, callbacks=[es], verbose=0)
        else:
            model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                      epochs=200, batch_size=32, callbacks=[es], verbose=0)
        all_preds.append(model.predict(X_test, verbose=0).ravel())

    # Average predictions across runs
    avg_preds_s = np.mean(all_preds, axis=0)
    avg_preds_ret = scaler_y.inverse_transform(avg_preds_s.reshape(-1,1)).ravel()
    pred_dir = np.sign(avg_preds_ret)

    preds_price = test_prices * (1 + avg_preds_ret)
    actual_price = test_prices * (1 + test_returns)
    rmse = np.sqrt(np.mean((preds_price - actual_price)**2))
    da = np.mean(pred_dir == actual_direction) * 100
    n_correct = int(round(da / 100 * len(X_test)))
    p_val = binomtest(n_correct, len(X_test), 0.5, alternative='greater').pvalue

    said_up_went_down = np.sum((pred_dir == 1) & (actual_direction == -1))
    said_down_went_up = np.sum((pred_dir == -1) & (actual_direction == 1))

    print(f"  {name:<45} | ${rmse:>5.2f} | {da:>5.1f}% | {p_val:.3f} | up→down:{said_up_went_down:>3} | down→up:{said_down_went_up:>3}")
    return avg_preds_ret

# 1. BASELINE (no augmentation)
print("\n  --- BASELINES ---")
train_and_eval(X_train, y_train, "No augmentation (baseline)")

# 2. DIRECTIONAL CLASS WEIGHTING
# Give higher weight to down days so model pays more attention
print("\n  --- CLASS WEIGHTING (penalise wrong direction more) ---")
weights_2x = np.ones(len(y_train))
down_mask = y_train_raw <= 0
weights_2x[down_mask] = 2.0
train_and_eval(X_train, y_train, "Down-day weight 2x", sample_weights=weights_2x)

weights_3x = np.ones(len(y_train))
weights_3x[down_mask] = 3.0
train_and_eval(X_train, y_train, "Down-day weight 3x", sample_weights=weights_3x)

weights_5x = np.ones(len(y_train))
weights_5x[down_mask] = 5.0
train_and_eval(X_train, y_train, "Down-day weight 5x", sample_weights=weights_5x)

# 3. OVERSAMPLE DOWN DAYS
print("\n  --- OVERSAMPLING DOWN DAYS ---")
down_idx = np.where(y_train_raw <= 0)[0]
up_idx = np.where(y_train_raw > 0)[0]

# Duplicate down days to balance
np.random.seed(42)
extra_down = np.random.choice(down_idx, size=len(up_idx) - len(down_idx), replace=True)
balanced_idx = np.concatenate([np.arange(len(y_train)), extra_down])
np.random.shuffle(balanced_idx)
X_train_balanced = X_train[balanced_idx]
y_train_balanced = y_train[balanced_idx]
train_and_eval(X_train_balanced, y_train_balanced, "Oversampled down days (balanced 50/50)")

# Extra oversampling
extra_down_2x = np.random.choice(down_idx, size=len(up_idx), replace=True)
over_idx = np.concatenate([np.arange(len(y_train)), extra_down_2x])
np.random.shuffle(over_idx)
X_train_over = X_train[over_idx]
y_train_over = y_train[over_idx]
train_and_eval(X_train_over, y_train_over, "Oversampled down days (60% down)")

# 4. JITTER ON DOWN DAYS ONLY
print("\n  --- TARGETED AUGMENTATION (augment down days only) ---")
X_down = X_train[down_idx]
y_down = y_train[down_idx]

# Jitter down days
X_down_jit = X_down + np.random.normal(0, 0.03, X_down.shape)
X_aug_down = np.concatenate([X_train, X_down_jit], axis=0)
y_aug_down = np.concatenate([y_train, y_down], axis=0)
train_and_eval(X_aug_down, y_aug_down, "Jitter on down days only (sigma=0.03)")

# Jitter + magnitude warp on down days
scales = np.random.normal(1.0, 0.1, (len(X_down), 1, X_down.shape[2]))
X_down_warp = X_down * scales
X_down_jit2 = X_down + np.random.normal(0, 0.05, X_down.shape)
X_aug_down2 = np.concatenate([X_train, X_down_jit, X_down_warp, X_down_jit2], axis=0)
y_aug_down2 = np.concatenate([y_train, y_down, y_down, y_down], axis=0)
train_and_eval(X_aug_down2, y_aug_down2, "Jitter+warp on down days (3x down data)")

# 5. SIGN-FLIP AUGMENTATION
print("\n  --- SIGN-FLIP AUGMENTATION ---")
# Flip the sign of returns to create synthetic down days from up days
X_flipped = X_train.copy()
X_flipped[:, :, 0] = -X_flipped[:, :, 0]  # Flip daily_return
X_flipped[:, :, 1] = -X_flipped[:, :, 1]  # Flip ret_lag_2
X_flipped[:, :, 2] = -X_flipped[:, :, 2]  # Flip ret_lag_5
X_flipped[:, :, 3] = -X_flipped[:, :, 3]  # Flip ret_lag_10
y_flipped = -y_train  # Flip target

X_aug_flip = np.concatenate([X_train, X_flipped], axis=0)
y_aug_flip = np.concatenate([y_train, y_flipped], axis=0)
train_and_eval(X_aug_flip, y_aug_flip, "Sign-flip augmentation (2x data, balanced)")

# Only flip up-day samples to create more down days
X_up_flipped = X_train[up_idx].copy()
X_up_flipped[:, :, 0] = -X_up_flipped[:, :, 0]
X_up_flipped[:, :, 1] = -X_up_flipped[:, :, 1]
X_up_flipped[:, :, 2] = -X_up_flipped[:, :, 2]
X_up_flipped[:, :, 3] = -X_up_flipped[:, :, 3]
y_up_flipped = -y_train[up_idx]

X_aug_flip2 = np.concatenate([X_train, X_up_flipped], axis=0)
y_aug_flip2 = np.concatenate([y_train, y_up_flipped], axis=0)
train_and_eval(X_aug_flip2, y_aug_flip2, "Sign-flip up→down only (balanced)")

# 6. CUSTOM LOSS: DIRECTIONAL PENALTY
print("\n  --- CUSTOM LOSS: DIRECTIONAL MSE ---")

def directional_mse(y_true, y_pred):
    """MSE with extra penalty when predicted direction is wrong"""
    mse = tf.square(y_true - y_pred)
    # Penalty when signs differ
    sign_match = tf.sign(y_true) * tf.sign(y_pred)
    # When signs differ, sign_match < 0, penalty multiplier = 3
    # When signs match, sign_match > 0, penalty multiplier = 1
    penalty = tf.where(sign_match < 0, 3.0, 1.0)
    return tf.reduce_mean(mse * penalty)

for penalty_name, loss_fn in [("Direction penalty 3x", directional_mse)]:
    all_preds = []
    for run in range(5):
        seed = 100 + run
        tf.random.set_seed(seed)
        np.random.seed(seed)
        model = make_model()
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss=loss_fn)
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=200, batch_size=32, callbacks=[es], verbose=0)
        all_preds.append(model.predict(X_test, verbose=0).ravel())

    avg_preds_s = np.mean(all_preds, axis=0)
    avg_preds_ret = scaler_y.inverse_transform(avg_preds_s.reshape(-1,1)).ravel()
    pred_dir = np.sign(avg_preds_ret)
    preds_price = test_prices * (1 + avg_preds_ret)
    actual_price = test_prices * (1 + test_returns)
    rmse = np.sqrt(np.mean((preds_price - actual_price)**2))
    da = np.mean(pred_dir == actual_direction) * 100
    n_correct = int(round(da / 100 * len(X_test)))
    p_val = binomtest(n_correct, len(X_test), 0.5, alternative='greater').pvalue
    said_up_went_down = np.sum((pred_dir == 1) & (actual_direction == -1))
    said_down_went_up = np.sum((pred_dir == -1) & (actual_direction == 1))
    print(f"  {penalty_name:<45} | ${rmse:>5.2f} | {da:>5.1f}% | {p_val:.3f} | up→down:{said_up_went_down:>3} | down→up:{said_down_went_up:>3}")

# 7. ENSEMBLE AVERAGE
print("\n  --- ENSEMBLE: Average of 5 diverse models ---")
all_model_preds = []
architectures = [
    (64, 32), (128, 64), (32, 16), (64, 64), (96, 48)
]
for units1, units2 in architectures:
    tf.random.set_seed(42)
    np.random.seed(42)
    inp = Input(shape=(WINDOW, len(FEATURES)))
    x = LSTM(units1, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
    x = Dropout(0.3)(x)
    x = LSTM(units2, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
    x = Dropout(0.3)(x)
    out = Dense(1)(x)
    model = Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='mse')
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=200, batch_size=32, callbacks=[es], verbose=0)
    all_model_preds.append(model.predict(X_test, verbose=0).ravel())

ensemble_preds_s = np.mean(all_model_preds, axis=0)
ensemble_preds_ret = scaler_y.inverse_transform(ensemble_preds_s.reshape(-1,1)).ravel()
pred_dir = np.sign(ensemble_preds_ret)
preds_price = test_prices * (1 + ensemble_preds_ret)
actual_price = test_prices * (1 + test_returns)
rmse = np.sqrt(np.mean((preds_price - actual_price)**2))
da = np.mean(pred_dir == actual_direction) * 100
n_correct = int(round(da / 100 * len(X_test)))
p_val = binomtest(n_correct, len(X_test), 0.5, alternative='greater').pvalue
said_up_went_down = np.sum((pred_dir == 1) & (actual_direction == -1))
said_down_went_up = np.sum((pred_dir == -1) & (actual_direction == 1))
print(f"  {'Ensemble 5 architectures':<45} | ${rmse:>5.2f} | {da:>5.1f}% | {p_val:.3f} | up→down:{said_up_went_down:>3} | down→up:{said_down_went_up:>3}")

print(f"\n{'='*110}")
print(f"  ALL AUGMENTATION EXPERIMENTS COMPLETE")
print(f"{'='*110}")

[*********************100%***********************]  1 of 1 completed


Dataset: 3238 days | Train: 2236 | Val: 456 | Test: 456
Training set: 1188 up days (53.1%), 1048 down days (46.9%)
Test set: 258 up (56.6%), 198 down (43.4%)

  ADVANCED DATA AUGMENTATION TO REDUCE WRONG CLASSIFICATIONS
  Method                                        |   RMSE |     DA | p-val |               Wrong Types
  ---------------------------------------------------------------------------------------------------------

  --- BASELINES ---
  No augmentation (baseline)                    | $ 3.77 |  56.4% | 0.004 | up→down:195 | down→up:  3

  --- CLASS WEIGHTING (penalise wrong direction more) ---
  Down-day weight 2x                            | $ 3.87 |  43.4% | 0.998 | up→down:  1 | down→up:256
  Down-day weight 3x                            | $ 4.01 |  43.2% | 0.998 | up→down:  0 | down→up:258
  Down-day weight 5x                            | $ 4.23 |  43.2% | 0.998 | up→down:  0 | down→up:258

  --- OVERSAMPLING DOWN DAYS ---
  Oversampled down days (balanced 50/50)        

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import yfinance as yf
from scipy.stats import binomtest
import warnings
warnings.filterwarnings('ignore')

df = yf.download('AAPL', start='2013-01-01', end='2025-12-31')
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df['daily_return'] = df['Close'].pct_change(1)
df['ret_lag_2'] = df['Close'].pct_change(2)
df['ret_lag_5'] = df['Close'].pct_change(5)
df['ret_lag_10'] = df['Close'].pct_change(10)
df['close_ma10_ratio'] = df['Close'] / df['Close'].rolling(10).mean() - 1
df['close_ma30_ratio'] = df['Close'] / df['Close'].rolling(30).mean() - 1
df['volatility_10'] = df['Close'].pct_change(1).rolling(10).std()
df['volatility_30'] = df['Close'].pct_change(1).rolling(30).std()
df['vol_change'] = df['Volume'].pct_change(1)
df['oc_gap'] = (df['Close'] - df['Open']) / df['Open']
df['target_return'] = df['Close'].pct_change(1).shift(-1)
df.dropna(inplace=True)

FEATURES = ['daily_return','ret_lag_2','ret_lag_5','ret_lag_10',
            'close_ma10_ratio','close_ma30_ratio','volatility_10','volatility_30',
            'vol_change','oc_gap']
WINDOW = 30

X_all = df[FEATURES].values
y_all = df['target_return'].values
prices = df['Close'].values

n = len(X_all)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

scaler_X = StandardScaler().fit(X_all[:train_end])

X_tr_s = scaler_X.transform(X_all[:train_end])
X_va_s = scaler_X.transform(X_all[train_end:val_end])
X_te_s = scaler_X.transform(X_all[val_end:])

# Binary labels: 1 = up, 0 = down
y_tr_bin = (y_all[:train_end] > 0).astype(np.float32)
y_va_bin = (y_all[train_end:val_end] > 0).astype(np.float32)
y_te_bin = (y_all[val_end:] > 0).astype(np.float32)

def create_sequences(X, y, window):
    Xs, ys = [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train, y_train = create_sequences(X_tr_s, y_tr_bin, WINDOW)
X_val, y_val = create_sequences(X_va_s, y_va_bin, WINDOW)
X_test, y_test = create_sequences(X_te_s, y_te_bin, WINDOW)

test_returns = y_all[val_end + WINDOW:]
test_prices = prices[val_end + WINDOW:]
actual_direction = (test_returns > 0).astype(int)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Train up: {np.sum(y_train):.0f} ({np.mean(y_train)*100:.1f}%) | down: {len(y_train)-np.sum(y_train):.0f} ({(1-np.mean(y_train))*100:.1f}%)")
print(f"Test  up: {np.sum(actual_direction)} ({np.mean(actual_direction)*100:.1f}%) | down: {len(actual_direction)-np.sum(actual_direction)} ({(1-np.mean(actual_direction))*100:.1f}%)")

print(f"\n{'='*100}")
print(f"  CLASSIFICATION VS REGRESSION COMPARISON")
print(f"{'='*100}")

def print_result(name, pred_dir, actual_dir):
    da = np.mean(pred_dir == actual_dir) * 100
    n_correct = int(round(da / 100 * len(actual_dir)))
    p_val = binomtest(n_correct, len(actual_dir), 0.5, alternative='greater').pvalue
    said_up_wrong = np.sum((pred_dir == 1) & (actual_dir == 0))
    said_down_wrong = np.sum((pred_dir == 0) & (actual_dir == 1))
    print(f"  {name:<50} | {da:>5.1f}% | p={p_val:.4f} | up→down:{said_up_wrong:>3} | down→up:{said_down_wrong:>3}")

header = f"  {'Model':<50} | {'DA':>6} | {'p-val':>8} | {'Wrong Types':>25}"
print(header)
print("  " + "-"*95)

# ============================================================
# 1. CLASSIFICATION: Binary cross-entropy
# ============================================================
print("\n  --- BINARY CLASSIFICATION (sigmoid + BCE) ---")

for units_name, u1, u2 in [("LSTM 64/32", 64, 32), ("LSTM 128/64", 128, 64), ("LSTM 32/16", 32, 16)]:
    all_preds = []
    for run in range(5):
        tf.random.set_seed(100+run)
        np.random.seed(100+run)
        inp = Input(shape=(WINDOW, len(FEATURES)))
        x = LSTM(u1, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
        x = Dropout(0.3)(x)
        x = LSTM(u2, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
        x = Dropout(0.3)(x)
        out = Dense(1, activation='sigmoid')(x)
        model = Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='binary_crossentropy', metrics=['accuracy'])
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=200, batch_size=32, callbacks=[es], verbose=0)
        preds = model.predict(X_test, verbose=0).ravel()
        all_preds.append(preds)
    avg_prob = np.mean(all_preds, axis=0)
    pred_dir = (avg_prob >= 0.5).astype(int)
    print_result(f"Classification {units_name}", pred_dir, actual_direction)

# ============================================================
# 2. CLASSIFICATION WITH CLASS WEIGHTS
# ============================================================
print("\n  --- CLASSIFICATION WITH CLASS WEIGHTS ---")
n_up = np.sum(y_train)
n_down = len(y_train) - n_up
weight_for_0 = len(y_train) / (2 * n_down)
weight_for_1 = len(y_train) / (2 * n_up)

for w_mult, w_name in [(1.0, "auto balanced"), (1.5, "down 1.5x"), (2.0, "down 2x")]:
    class_w = {0: weight_for_0 * w_mult, 1: weight_for_1}
    all_preds = []
    for run in range(5):
        tf.random.set_seed(100+run)
        np.random.seed(100+run)
        inp = Input(shape=(WINDOW, len(FEATURES)))
        x = LSTM(64, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
        x = Dropout(0.3)(x)
        x = LSTM(32, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
        x = Dropout(0.3)(x)
        out = Dense(1, activation='sigmoid')(x)
        model = Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='binary_crossentropy')
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model.fit(X_train, y_train, class_weight=class_w, validation_data=(X_val, y_val),
                  epochs=200, batch_size=32, callbacks=[es], verbose=0)
        preds = model.predict(X_test, verbose=0).ravel()
        all_preds.append(preds)
    avg_prob = np.mean(all_preds, axis=0)
    pred_dir = (avg_prob >= 0.5).astype(int)
    print_result(f"Classification {w_name}", pred_dir, actual_direction)

# ============================================================
# 3. CLASSIFICATION WITH THRESHOLD TUNING
# ============================================================
print("\n  --- THRESHOLD TUNING (adjust decision boundary) ---")
# Use best model from above
all_preds = []
for run in range(5):
    tf.random.set_seed(100+run)
    np.random.seed(100+run)
    inp = Input(shape=(WINDOW, len(FEATURES)))
    x = LSTM(64, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
    x = Dropout(0.3)(x)
    x = LSTM(32, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
    x = Dropout(0.3)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='binary_crossentropy')
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=200, batch_size=32, callbacks=[es], verbose=0)
    all_preds.append(model.predict(X_test, verbose=0).ravel())

avg_prob = np.mean(all_preds, axis=0)

for threshold in [0.40, 0.45, 0.50, 0.55, 0.60]:
    pred_dir = (avg_prob >= threshold).astype(int)
    print_result(f"Threshold = {threshold}", pred_dir, actual_direction)

# ============================================================
# 4. FOCAL LOSS (focus on hard examples)
# ============================================================
print("\n  --- FOCAL LOSS (focus on hard-to-classify days) ---")

def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        loss = -alpha_t * tf.pow(1 - p_t, gamma) * tf.math.log(p_t)
        return tf.reduce_mean(loss)
    return focal_loss_fn

for gamma, alpha in [(2.0, 0.25), (2.0, 0.5), (3.0, 0.5), (2.0, 0.75)]:
    all_preds = []
    for run in range(5):
        tf.random.set_seed(100+run)
        np.random.seed(100+run)
        inp = Input(shape=(WINDOW, len(FEATURES)))
        x = LSTM(64, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
        x = Dropout(0.3)(x)
        x = LSTM(32, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
        x = Dropout(0.3)(x)
        out = Dense(1, activation='sigmoid')(x)
        model = Model(inp, out)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss=focal_loss(gamma, alpha))
        es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        model.fit(X_train, y_train, validation_data=(X_val, y_val),
                  epochs=200, batch_size=32, callbacks=[es], verbose=0)
        all_preds.append(model.predict(X_test, verbose=0).ravel())
    avg_prob = np.mean(all_preds, axis=0)
    pred_dir = (avg_prob >= 0.5).astype(int)
    print_result(f"Focal loss gamma={gamma} alpha={alpha}", pred_dir, actual_direction)

# ============================================================
# 5. COMBINED: Classification + External Features
# ============================================================
print("\n  --- CLASSIFICATION + EXTERNAL FEATURES ---")

vix = yf.download('^VIX', start='2013-01-01', end='2025-12-31')
sp500 = yf.download('^GSPC', start='2013-01-01', end='2025-12-31')
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)
if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

df2 = df.copy()
df2['VIX'] = vix['Close']
df2['VIX_change'] = vix['Close'].pct_change(1)
df2['SP500_ret'] = sp500['Close'].pct_change(1)
df2.dropna(inplace=True)

FEATURES_EXT = FEATURES + ['VIX', 'VIX_change', 'SP500_ret']
X_all2 = df2[FEATURES_EXT].values
y_all2 = df2['target_return'].values

n2 = len(X_all2)
te2 = int(n2 * 0.70)
ve2 = int(n2 * 0.85)

scaler_X2 = StandardScaler().fit(X_all2[:te2])
X_tr2 = scaler_X2.transform(X_all2[:te2])
X_va2 = scaler_X2.transform(X_all2[te2:ve2])
X_te2 = scaler_X2.transform(X_all2[ve2:])
y_tr2 = (y_all2[:te2] > 0).astype(np.float32)
y_va2 = (y_all2[te2:ve2] > 0).astype(np.float32)
actual_dir2 = (y_all2[ve2:] > 0).astype(int)

Xtr2, ytr2 = create_sequences(X_tr2, y_tr2, WINDOW)
Xva2, yva2 = create_sequences(X_va2, y_va2, WINDOW)
Xte2, yte2 = create_sequences(X_te2, (y_all2[ve2:] > 0).astype(np.float32), WINDOW)
adir2 = actual_dir2[WINDOW:]

all_preds = []
for run in range(5):
    tf.random.set_seed(100+run)
    np.random.seed(100+run)
    inp = Input(shape=(WINDOW, len(FEATURES_EXT)))
    x = LSTM(64, return_sequences=True, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(inp)
    x = Dropout(0.3)(x)
    x = LSTM(32, kernel_regularizer=l2(0.005), recurrent_regularizer=l2(0.005))(x)
    x = Dropout(0.3)(x)
    out = Dense(1, activation='sigmoid')(x)
    model = Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss='binary_crossentropy')
    es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(Xtr2, ytr2, validation_data=(Xva2, yva2),
              epochs=200, batch_size=32, callbacks=[es], verbose=0)
    all_preds.append(model.predict(Xte2, verbose=0).ravel())

avg_prob = np.mean(all_preds, axis=0)
pred_dir = (avg_prob >= 0.5).astype(int)
print_result(f"Classification + VIX/SP500 (13 features)", pred_dir, adir2)

print(f"\n{'='*100}")
print(f"  ALL CLASSIFICATION EXPERIMENTS COMPLETE")
print(f"{'='*100}")

[*********************100%***********************]  1 of 1 completed


Train: 2236 | Val: 456 | Test: 456
Train up: 1188 (53.1%) | down: 1048 (46.9%)
Test  up: 258 (56.6%) | down: 198 (43.4%)

  CLASSIFICATION VS REGRESSION COMPARISON
  Model                                              |     DA |    p-val |               Wrong Types
  -----------------------------------------------------------------------------------------------

  --- BINARY CLASSIFICATION (sigmoid + BCE) ---
  Classification LSTM 64/32                          |  56.6% | p=0.0028 | up→down:198 | down→up:  0
  Classification LSTM 128/64                         |  56.6% | p=0.0028 | up→down:198 | down→up:  0
  Classification LSTM 32/16                          |  56.6% | p=0.0028 | up→down:198 | down→up:  0

  --- CLASSIFICATION WITH CLASS WEIGHTS ---
  Classification auto balanced                       |  50.9% | p=0.3715 | up→down:123 | down→up:101
  Classification down 1.5x                           |  43.4% | p=0.9979 | up→down:  0 | down→up:258
  Classification down 2x              

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


  Classification + VIX/SP500 (13 features)           |  56.6% | p=0.0028 | up→down:198 | down→up:  0

  ALL CLASSIFICATION EXPERIMENTS COMPLETE
